In [63]:
# # Simple duplicate detection

import pandas as pd

df = pd.DataFrame({
    'Name': ['John Smith', 'Jane Doe', 'John Smith', 'Jon Smith'],
    'Age': [30, 25, 30, 30],
    'City': ['New York', 'Boston', 'New York', 'New York']
})

print(df.duplicated().sum())

1


In [64]:
# Exact Duplicates

import pandas as pd

df = pd.DataFrame({
    'Name': ['John Doe', 'Jane Smith', 'John Doe'],
    'Age': [30, 25, 30],
    'City': ['New York', 'Boston', 'New York']
})
df

,Name,Age,City
0,John Doe,30,New York
1,Jane Smith,25,Boston
2,John Doe,30,New York


In [65]:
df.drop_duplicates()

,Name,Age,City
0,John Doe,30,New York
1,Jane Smith,25,Boston


In [66]:
df.duplicated()

0    False
1    False
2     True
dtype: bool

In [67]:
# Partial Exact Duplicate

df = pd.DataFrame({
    'Name': ['John Doe', 'John Doe', 'Jane Smith'],
    'Age': [30, 30, 25],
    'City': ['New York', 'Boston', 'Boston']
})

df.drop_duplicates()

,Name,Age,City
0,John Doe,30,New York
1,John Doe,30,Boston
2,Jane Smith,25,Boston


In [68]:
df.drop_duplicates(subset=['Name', 'Age'])

,Name,Age,City
0,John Doe,30,New York
2,Jane Smith,25,Boston


In [69]:
# Near Duplicate(Fuzzy Duplicate)

df = pd.DataFrame({
    'Name': ['John Smith', 'Jon Smyth'],
    'Age': [30, 30],
    'City': ['New York', 'NY']
})
df.drop_duplicates(subset=['Name', 'Age'])

,Name,Age,City
0,John Smith,30,New York
1,Jon Smyth,30,NY


In [70]:
# Cross-Column Duplicate

df = pd.DataFrame({
    'Full_Name': ['John Doe', 'Jane Smith'],
    'First_Name': ['John', 'Jane'],
    'Last_Name': ['Doe', 'Smith']
})
df

,Full_Name,First_Name,Last_Name
0,John Doe,John,Doe
1,Jane Smith,Jane,Smith


In [71]:
df.drop_duplicates(subset=['Full_Name'])

,Full_Name,First_Name,Last_Name
0,John Doe,John,Doe
1,Jane Smith,Jane,Smith


In [72]:
# Time-Based Duplicate

df = pd.DataFrame({
    'User': ['User1', 'User1', 'User2'],
    'Action': ['Login', 'Login', 'Logout'],
    'Timestamp': ['2023-05-01 10:00:00', '2023-05-01 10:00:01', '2023-05-01 11:00:00']
})
df

,User,Action,Timestamp
0,User1,Login,2023-05-01 10:00:00
1,User1,Login,2023-05-01 10:00:01
2,User2,Logout,2023-05-01 11:00:00


### Semantic Duplicate

Examples:
- "IBM" and "International Business Machines"
- "H2O" and "Water"

In [73]:
# Exact Duplicates

import pandas as pd

df = pd.DataFrame({
    'Name': ['John Doe', 'Jane Smith', 'John Doe', 'Alice Johnson', 'John Doe'],
    'Age': [30, 25, 30, 35, 30],
    'City': ['New York', 'Boston', 'New York', 'Chicago', 'Los Angeles']
})
df

,Name,Age,City
0,John Doe,30,New York
1,Jane Smith,25,Boston
2,John Doe,30,New York
3,Alice Johnson,35,Chicago
4,John Doe,30,Los Angeles


In [74]:
# Identify exact duplicates

exact_duplicates = df[df.duplicated()]
print("Exact Duplicates:")
exact_duplicates

Exact Duplicates:


,Name,Age,City
2,John Doe,30,New York


In [75]:
Not_duplicates = df[~df.duplicated(subset=['Name','Age'])]
print("Exact Duplicates:")
Not_duplicates

Exact Duplicates:


,Name,Age,City
0,John Doe,30,New York
1,Jane Smith,25,Boston
3,Alice Johnson,35,Chicago


In [76]:
df.drop_duplicates(subset=['Name','Age'],inplace=True)
df

,Name,Age,City
0,John Doe,30,New York
1,Jane Smith,25,Boston
3,Alice Johnson,35,Chicago


In [77]:
# Identify duplicates based on Name and Age

df = pd.DataFrame({
    'Name': ['John Doe', 'Jane Smith', 'John Doe', 'Alice Johnson', 'John Doe'],
    'Age': [30, 25, 30, 35, 30],
    'City': ['New York', 'Boston', 'New York', 'Chicago', 'Los Angeles']
})
df

subset_duplicates = df[df.duplicated(subset=['Name', 'Age'], keep=False)]

print("Subset Duplicates (Name and Age):")
subset_duplicates

Subset Duplicates (Name and Age):


,Name,Age,City
0,John Doe,30,New York
2,John Doe,30,New York
4,John Doe,30,Los Angeles


In [78]:
# Fuzzy Matching

%pip install thefuzz

Note: you may need to restart the kernel to use updated packages.


In [79]:
from thefuzz import fuzz

In [80]:
fuzz.ratio('John Doe', 'John D')

86

In [81]:
fuzz.partial_ratio('John Doe', 'John D')

100

In [82]:
fuzz.partial_ratio('John Doe', 'Doe John')

67

In [83]:
fuzz.token_sort_ratio('John Doe', 'Doe John')

100

In [84]:
import itertools

list(itertools.combinations([1,2,3],2))

[(1, 2), (1, 3), (2, 3)]

In [85]:
import itertools

def fuzzy_dedupe(df, column, threshold=80):
    duplicates = []
    for (idx1, row1), (idx2, row2) in itertools.combinations(df.iterrows(), 2):
        sim_ratio = fuzz.ratio(row1[column], row2[column])
        print(sim_ratio)
        if fuzz.ratio(row1[column], row2[column]) >= threshold:
            duplicates.append((idx1, idx2))
    return duplicates

fuzzy_dupes = fuzzy_dedupe(df, 'Name')
fuzzy_dupes

33
100
48
100
33
26
33
48
100
48


[(0, 2), (0, 4), (2, 4)]

In [86]:
df

,Name,Age,City
0,John Doe,30,New York
1,Jane Smith,25,Boston
2,John Doe,30,New York
3,Alice Johnson,35,Chicago
4,John Doe,30,Los Angeles


In [87]:
print("Fuzzy Duplicates:")
for idx1, idx2 in fuzzy_dupes:
    print("Potential duplicate:")
    print()
    print(df.loc[idx1])
    print()
    print(df.loc[idx2])
    print('-' * 50)

Fuzzy Duplicates:
Potential duplicate:

Name    John Doe
Age           30
City    New York
Name: 0, dtype: object

Name    John Doe
Age           30
City    New York
Name: 2, dtype: object
--------------------------------------------------
Potential duplicate:

Name    John Doe
Age           30
City    New York
Name: 0, dtype: object

Name       John Doe
Age              30
City    Los Angeles
Name: 4, dtype: object
--------------------------------------------------
Potential duplicate:

Name    John Doe
Age           30
City    New York
Name: 2, dtype: object

Name       John Doe
Age              30
City    Los Angeles
Name: 4, dtype: object
--------------------------------------------------


In [88]:
# Phonetic matching

%pip install jellyfish

Note: you may need to restart the kernel to use updated packages.


In [89]:
import jellyfish

jellyfish.soundex('John Doe'), jellyfish.soundex('John Doa')

('J530', 'J530')

In [90]:

jellyfish.soundex('Kane'), jellyfish.soundex('Jane')

('K500', 'J500')

In [91]:
import jellyfish

# data with phonetic duplicates

df = pd.DataFrame({
    'Name': ['John Doe', 'Jane Smith', 'John Doh', 'Alice Johnson', 'John Doe'],
    'Age': [30, 25, 30, 35, 30],
    'City': ['New York', 'Boston', 'New York', 'Chicago', 'Los Angeles']
})

def phonetic_match(df, column):
    df['phonetic'] = df[column].apply(jellyfish.soundex)
    return df[df.duplicated(subset='phonetic', keep=False)].sort_values('phonetic')

phonetic_dupes = phonetic_match(df, 'Name')
print("Phonetic Duplicates:")
print(phonetic_dupes)

Phonetic Duplicates:
       Name  Age         City phonetic
0  John Doe   30     New York     J530
2  John Doh   30     New York     J530
4  John Doe   30  Los Angeles     J530


In [92]:
# Distance Based Method

from scipy.spatial.distance import pdist, squareform
import numpy as np
import pandas as pd

def distance_based_duplicates(df, columns, threshold=5):
    subset = df[columns]
    dist_matrix = pdist(subset.values, metric='euclidean')
    dist_matrix = squareform(dist_matrix)
    print(f'dist_matrix shape: {dist_matrix.shape}')
    print(dist_matrix)
    print('-' * 50)
    duplicate_pairs = np.argwhere((dist_matrix > 0) & (dist_matrix < threshold))
    return duplicate_pairs

df = pd.DataFrame({
    'Name': ['John Doe', 'Jane Smith', 'John Doe', 'Alice Johnson', 'John Doe'],
    'Age': [32, 26, 29, 31, 28],
    'Height': [170, 165, 170, 168, 170],
    'Weight': [60, 55, 60, 58, 60]
})
df

,Name,Age,Height,Weight
0,John Doe,32,170,60
1,Jane Smith,26,165,55
2,John Doe,29,170,60
3,Alice Johnson,31,168,58
4,John Doe,28,170,60


In [93]:
subset = df[['Age', 'Height', 'Weight']]

In [94]:
subset.values

array([[ 32, 170,  60],
       [ 26, 165,  55],
       [ 29, 170,  60],
       [ 31, 168,  58],
       [ 28, 170,  60]])

In [95]:
from scipy.spatial.distance import pdist, squareform
import numpy as np

pdist(subset.values)

array([9.2736185 , 3.        , 3.        , 4.        , 7.68114575,
       6.55743852, 7.34846923, 3.46410162, 1.        , 4.12310563])

In [96]:
pdist(subset.values, metric='euclidean')

array([9.2736185 , 3.        , 3.        , 4.        , 7.68114575,
       6.55743852, 7.34846923, 3.46410162, 1.        , 4.12310563])

In [97]:
squareform(pdist(subset.values, metric='euclidean'))

array([[0.        , 9.2736185 , 3.        , 3.        , 4.        ],
       [9.2736185 , 0.        , 7.68114575, 6.55743852, 7.34846923],
       [3.        , 7.68114575, 0.        , 3.46410162, 1.        ],
       [3.        , 6.55743852, 3.46410162, 0.        , 4.12310563],
       [4.        , 7.34846923, 1.        , 4.12310563, 0.        ]])

In [98]:
# Only use numeric columns for distance calculation

numeric_columns = ['Age','Height', 'Weight']
dist_dupes = distance_based_duplicates(df, numeric_columns, threshold=3)

print("Distance-Based Duplicates:")
if len(dist_dupes) > 0:
    print(f'{len(dist_dupes)} duplicates found')
    for pair in dist_dupes:
        print(df.iloc[pair])
        print("-" * 50)
else:
    print("No duplicates found.")

dist_matrix shape: (5, 5)
[[0.         9.2736185  3.         3.         4.        ]
 [9.2736185  0.         7.68114575 6.55743852 7.34846923]
 [3.         7.68114575 0.         3.46410162 1.        ]
 [3.         6.55743852 3.46410162 0.         4.12310563]
 [4.         7.34846923 1.         4.12310563 0.        ]]
--------------------------------------------------
Distance-Based Duplicates:
2 duplicates found
       Name  Age  Height  Weight
2  John Doe   29     170      60
4  John Doe   28     170      60
--------------------------------------------------
       Name  Age  Height  Weight
4  John Doe   28     170      60
2  John Doe   29     170      60
--------------------------------------------------


In [99]:
# Machine Learning Approaches for Duplicate Detection

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from thefuzz import fuzz

# Create a sample dataset

np.random.seed(42)
data = {
    'Name': ['John Smith', 'Jane Doe', 'Jon Smyth', 'Jane Doe', 'John Smyth', 'Janet Doe'],
    'Age': [29, 28, 31, 26, 28, 26],
    'City': ['New York', 'Boston', 'New York', 'Boston', 'NY', 'Bostan'],
    'Salary': [50000, 60000, 51000, 60000, 50000, 61000]
}

df = pd.DataFrame(data)
print("Sample Dataset:")
df

Sample Dataset:


,Name,Age,City,Salary
0,John Smith,29,New York,50000
1,Jane Doe,28,Boston,60000
2,Jon Smyth,31,New York,51000
3,Jane Doe,26,Boston,60000
4,John Smyth,28,NY,50000
5,Janet Doe,26,Bostan,61000


In [100]:
# Supervised Learning Approach

def create_pair_features(left, right):
    """Create features that capture the similarity between two records"""
    features = {}
    features['name_similarity'] = fuzz.ratio(left['Name'], right['Name']) / 100.0
    features['age_diff'] = abs(left['Age'] - right['Age'])
    features['city_similarity'] = fuzz.ratio(left['City'], right['City']) / 100.0
    features['salary_diff'] = abs(left['Salary'] - right['Salary'])
    return features

In [101]:
from faker import Faker
import random

# Create a sample dataset

fake = Faker()
num_records = 20
num_unique_values = 3

names = [fake.name() for _ in range(num_unique_values)]
age = [fake.random_int(min=20, max=40) for _ in range(num_unique_values)]
cities = [fake.city() for _ in range(num_unique_values)]
salaries = [fake.random_int(min=30000, max=100000) for _ in range(num_unique_values)]

names = [random.choice(names) for _ in range(num_records)]
age = [random.choice(age) for _ in range(num_records)]
cities = [random.choice(cities) for _ in range(num_records)]
salaries = [random.choice(salaries) for _ in range(num_records)]

In [102]:
# Create all possible pairs of records

pairs = []
labels = []
for i in range(num_records):
    left = {
        'Name': names[i],
        'Age': age[i],
        'City': cities[i],
        'Salary': salaries[i]
    }
    for j in range(i+1, num_records):
        right = {
            'Name': names[j],
            'Age': age[j],
            'City': cities[j],
            'Salary': salaries[j]
        }
        pairs.append(create_pair_features(left, right))
        
        # Label as duplicate if names are very similar and age difference is small
        
        is_duplicate = (fuzz.ratio(left['Name'], right['Name']) > 80) and (abs(left['Age'] - right['Age']) <= 1)
        labels.append(int(is_duplicate))

X = pd.DataFrame(pairs)
y = pd.Series(labels)

X

,name_similarity,age_diff,city_similarity,salary_diff
0,0.16,2,1.00,24057
1,0.16,0,1.00,0
2,0.29,2,0.38,0
3,1.00,2,0.38,0
4,0.29,2,1.00,24880
...,...,...,...,...
185,0.16,4,1.00,48937
186,1.00,4,1.00,0
187,0.16,4,1.00,48937
188,1.00,4,1.00,0


In [103]:
y

0      0
1      0
2      0
3      0
4      0
      ..
185    0
186    0
187    0
188    0
189    0
Length: 190, dtype: int64

In [104]:
y.sum()

np.int64(24)

In [105]:
# Split the data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train a Random Forest classifier

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate the model

y_pred = clf.predict(X_test)
print("\nSupervised Learning Results:")
print(classification_report(y_test, y_pred))


Supervised Learning Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00         5

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38



In [106]:
# Predict on all pairs

all_pairs_pred = clf.predict(X_test)
all_pairs_pred

array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0])

In [107]:
for i, pred in enumerate(all_pairs_pred):
    if pred == 1:
        print(X_test.iloc[i])
        print('-' * 50)

name_similarity        1.0
age_diff               0.0
city_similarity        1.0
salary_diff        24057.0
Name: 73, dtype: float64
--------------------------------------------------
name_similarity    1.00
age_diff           0.00
city_similarity    0.38
salary_diff        0.00
Name: 172, dtype: float64
--------------------------------------------------
name_similarity        1.0
age_diff               0.0
city_similarity        1.0
salary_diff        48937.0
Name: 105, dtype: float64
--------------------------------------------------
name_similarity    1.00
age_diff           0.00
city_similarity    0.38
salary_diff        0.00
Name: 100, dtype: float64
--------------------------------------------------
name_similarity    1.0
age_diff           0.0
city_similarity    1.0
salary_diff        0.0
Name: 160, dtype: float64
--------------------------------------------------


In [108]:
# Unsupervised Learning Approach

def unsupervised_duplicates(df, threshold=0.6):
    
    # Prepare the data
    
    numeric_cols = ['Age', 'Salary']
    categorical_cols = ['City', 'Name']

    # One-hot encode categorical variables
    
    df_encoded = pd.get_dummies(df, columns=categorical_cols)

    # Standardize numeric features
    
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded[numeric_cols]), columns=numeric_cols)

    # Combine scaled numeric and one-hot encoded features
    
    df_prepared = pd.concat([df_scaled, df_encoded.drop(columns=numeric_cols)], axis=1)

    # Compute similarity matrix
    
    sim_matrix = cosine_similarity(df_prepared)

    # Find potential duplicates
    
    potential_duplicates = []
    for i in range(len(df)):
        for j in range(i+1, len(df)):
            if sim_matrix[i][j] > threshold:
                potential_duplicates.append((i, j, sim_matrix[i][j]))

    return potential_duplicates

duplicates = unsupervised_duplicates(df)
print("\nPotential Duplicates (Unsupervised):")
for i, j, sim in duplicates:
    print(f"Record {i} and Record {j} might be duplicates (similarity: {sim:.2f})")
    print(df.iloc[i])
    print(df.iloc[j])
    print()


Potential Duplicates (Unsupervised):
Record 0 and Record 2 might be duplicates (similarity: 0.65)
Name      John Smith
Age               29
City        New York
Salary         50000
Name: 0, dtype: object
Name      Jon Smyth
Age              31
City       New York
Salary        51000
Name: 2, dtype: object

Record 1 and Record 3 might be duplicates (similarity: 0.83)
Name      Jane Doe
Age             28
City        Boston
Salary       60000
Name: 1, dtype: object
Name      Jane Doe
Age             26
City        Boston
Salary       60000
Name: 3, dtype: object



In [109]:
# hybrid approach 

def hybrid_duplicate_detection(df, unsupervised_threshold=0.25, supervised_threshold=0.5):
    
    # Step 1: Use unsupervised method to identify potential duplicates
    
    potential_dupes = unsupervised_duplicates(df, threshold=unsupervised_threshold)

    # Step 2: Create features for these potential duplicates
    
    X = []
    for i, j, _ in potential_dupes:
        X.append(create_pair_features(df.iloc[i], df.iloc[j]))
    X = pd.DataFrame(X)

    # Step 3: Use the supervised model to classify these potential duplicates
    
    probas = clf.predict_proba(X)[:, 1]  # Probability of being a duplicate

    # Step 4: Filter based on the supervised model's confidence
    
    final_duplicates = [(potential_dupes[i][0], potential_dupes[i][1])
                        for i, proba in enumerate(probas) if proba > supervised_threshold]

    return final_duplicates

hybrid_dupes = hybrid_duplicate_detection(df)
print("\nPotential Duplicates (Hybrid Approach):")
for i, j in hybrid_dupes:
    print(f"Record {i} and Record {j} might be duplicates")
    print(df.iloc[i])
    print(df.iloc[j])
    print()


Potential Duplicates (Hybrid Approach):
Record 0 and Record 4 might be duplicates
Name      John Smith
Age               29
City        New York
Salary         50000
Name: 0, dtype: object
Name      John Smyth
Age               28
City              NY
Salary         50000
Name: 4, dtype: object

Record 3 and Record 5 might be duplicates
Name      Jane Doe
Age             26
City        Boston
Salary       60000
Name: 3, dtype: object
Name      Janet Doe
Age              26
City         Bostan
Salary        61000
Name: 5, dtype: object



In [110]:
# Pseudocode for a combined approach

def identify_duplicates(df):
    exact_dupes = df[df.duplicated(keep=False)]
    remaining = df.drop(exact_dupes.index)

    fuzzy_dupes = pd.DataFrame(columns=df.columns)
    for col in ['Name', 'City']:
        fuzzy_dupes = pd.concat([fuzzy_dupes,
                                 remaining[remaining.index.isin(fuzzy_dedupe(remaining, col))]])

    remaining = remaining.drop(fuzzy_dupes.index)
    phonetic_dupes = phonetic_match(remaining, 'Name')

    return pd.concat([exact_dupes, fuzzy_dupes, phonetic_dupes])

all_duplicates = identify_duplicates(df)
print("\nAll Detected Duplicates:")
print(all_duplicates)

33
84
33
90
32
35
100
33
94
35
95
33
33
94
32
14
100
14
40
14
14
100
0
83
14
40
14
0
83
0

All Detected Duplicates:
         Name Age      City Salary phonetic
0  John Smith  29  New York  50000     J525
2   Jon Smyth  31  New York  51000     J525
4  John Smyth  28        NY  50000     J525
1    Jane Doe  28    Boston  60000     J530
3    Jane Doe  26    Boston  60000     J530


In [112]:
# Mitigation Strategies
# Example of how duplicates can affect basic statistics

import pandas as pd

df = pd.DataFrame({
    'Value': [1, 2, 3, 3, 4, 5]  # Note the duplicate '3'
})

print(f"Mean with duplicate: {df['Value'].mean()}")
print(f"Mean without duplicate: {df['Value'].drop_duplicates().mean()}")

Mean with duplicate: 3.0
Mean without duplicate: 3.0
